# <center>**Evaluación**</center>
## <center>*Gestión Empresarial*</center>
### <center>Ethan Bula y Cesia Burgos</center>

In [15]:
import pulp as p
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Datos del problema

tiempo = 11  # Número de semanas

Pronostico = [1000,2000,1000,4000,3000,2000,2000,1000,3000,7000,7000]

# Costos

Costo_Tela=[20]
for i in range(1,tiempo):
    if i < 8:
        Costo_Tela.append(Costo_Tela[i-1]*1.07)
    else:
        Costo_Tela.append(Costo_Tela[i-1])

Costo_Hilo=[10]
for i in range(1,tiempo):
    if i < 8:
        Costo_Hilo.append(Costo_Hilo[i-1]*1.07)
    else:
        Costo_Hilo.append(Costo_Hilo[i-1])

Costo_Lamina=[20]
for i in range(1,tiempo):
    if i < 8:
        Costo_Lamina.append(Costo_Lamina[i-1]*1.07)
    else:
        Costo_Lamina.append(Costo_Lamina[i-1])

Costo_Fabricar=[4]
for i in range(1,tiempo):
    if i < 8:
        Costo_Fabricar.append(Costo_Fabricar[i-1]*1.07)
    else:
        Costo_Fabricar.append(Costo_Fabricar[i-1])

Costo_Almacenar=[12] # Por m3
for i in range(1,tiempo):
    if i < 8:
        Costo_Almacenar.append(Costo_Almacenar[i-1]*1.07)
    else:
        Costo_Almacenar.append(Costo_Almacenar[i-1])

# Uso de materia prima por unidad de producto terminado
Uso_Tela=10
Uso_Hilo=10
Uso_Lamina=10

Requerimientos = [Uso_Tela, Uso_Hilo, Uso_Lamina]

# Tiempos [minutos]
Tiempo_Tela=4
Tiempo_Hilo=5
Tiempo_Lamina=12
Tiempo_Ensamble=40

Tiempos= [Tiempo_Tela, Tiempo_Hilo, Tiempo_Lamina, Tiempo_Ensamble]

Dias_Semana = 6
Horas_Dia = 12
Minutos_Por_Semana = Dias_Semana * Horas_Dia * 60  # 4320

# Cantidades
Corte_CNC =4
Corte_hilo=4
Corte_Plasma=10
Selladora=31

Cantidades = [Corte_CNC, Corte_hilo, Corte_Plasma, Selladora]

# Volumenes [m3]
Bodega1=108
Bodega2=108*1.15
Volumen_Tela=0.2
Volumen_Hilo=0.016
Volumen_Lamina=0.025
Volumen_PT=0.004

Volumenes = [Volumen_Tela, Volumen_Hilo, Volumen_Lamina, Volumen_PT]

Inventario_Inicial = [0, 0, 550, 0] # solo hay inventario inicial de lámina 

# Definir modelo
prob = p.LpProblem("Minimizar_Costo", p.LpMinimize)

# Variables de decisión
Tela = p.LpVariable.dicts("Tela", (range(tiempo)), lowBound=0, cat='Integer')
Hilo = p.LpVariable.dicts("Hilo", (range(tiempo)), lowBound=0, cat='Integer')
Lamina = p.LpVariable.dicts("Lamina", (range(tiempo)), lowBound=0, cat='Integer')
PT = p.LpVariable.dicts("PT", (range(tiempo)), lowBound=0, cat='Integer')
Inventario = p.LpVariable.dicts("Inventario", (range(4), range(tiempo)), lowBound=0, cat='Integer')


# Función objetivo
Costo_Compras=[]
for t in range(tiempo):
    Costo_Compras.append(Costo_Tela[t]*Tela[t] + Costo_Hilo[t]*Hilo[t] + Costo_Lamina[t]*Lamina[t])

Costo_Fabricacion=[]
for t in range(tiempo):
    Costo_Fabricacion.append(Costo_Fabricar[t]*PT[t])

Costo_Almacenamiento=[]
for t in range(tiempo):
    for i in range(4):
        cantidad = Tela[t] if i == 0 else Hilo[t] if i == 1 else Lamina[t] if i == 2 else PT[t]
        if t == 0:
            costo = Costo_Almacenar[t] * Volumenes[i] * (Inventario[i][t] + cantidad) / 2
        else:
            costo = Costo_Almacenar[t] * Volumenes[i] * (Inventario[i][t] + Inventario[i][t-1] + cantidad) / 2
        Costo_Almacenamiento.append(costo)


prob += p.lpSum(Costo_Compras) + p.lpSum(Costo_Fabricacion) + p.lpSum(Costo_Almacenamiento), "Costo_Total"

# Restricciones

## Balance de inventario de materia prima

leadtime_lamina = 2

for i in range(3):  # i = 0: Tela, 1: Hilo, 2: Lámina
    for t in range(tiempo):
        compra = Tela[t] if i == 0 else Hilo[t] if i == 1 else Lamina[t]
        consumo = PT[t] * (1 / Requerimientos[i])
        if i == 2:  # Solo la lámina tiene inventario inicial
            if t == 0:
                prob += Inventario[i][t] == Inventario_Inicial[i] - consumo, f"Balance_inventario_{i}_{t}"
            elif t < leadtime_lamina:
                prob += Inventario[i][t] == Inventario[i][t-1] - consumo, f"Balance_inventario_{i}_{t}"
            else:
                prob += Inventario[i][t] == Inventario[i][t-1] + Lamina[t-leadtime_lamina] - consumo, f"Balance_inventario_{i}_{t}"
        else: 
            if t == 0:
                prob += Inventario[i][t] == compra - consumo, f"Balance_inventario_{i}_{t}"
            else:
                prob += Inventario[i][t] == Inventario[i][t-1] + compra - consumo, f"Balance_inventario_{i}_{t}"

## Balance de producto terminado

for t in range(tiempo):
    if t == 0:
        inventario_inicial_PT = 0  # puedes cambiarlo si hay stock inicial
        prob += Inventario[3][t] == inventario_inicial_PT + PT[t] - Pronostico[t], f"Balance_PT_{t}"
    else:
        prob += Inventario[3][t] == Inventario[3][t-1] + PT[t] - Pronostico[t], f"Balance_PT_{t}"
    

## Restriccion de almacenamiento
espacio_maximo = [Bodega1 if t < 3 else Bodega2 for t in range(tiempo)]

for t in range(tiempo):
    espacio_ocupado = (
        Inventario[0][t] * Volumenes[0] +  # Tela
        Inventario[1][t] * Volumenes[1] +  # Hilo
        Inventario[2][t] * Volumenes[2] +  # Lámina
        Inventario[3][t] * Volumenes[3]    # Producto terminado
    )
    prob += espacio_ocupado <= espacio_maximo[t], f"Restriccion_espacio_{t}"

## Restricciones de capacidad de producción
for t in range(tiempo):
    for i in range(4):  # 0: Corte tela, 1: Corte hilo, 2: Corte lámina, 3: Ensamble
        tiempo_requerido = PT[t] * Tiempos[i]
        tiempo_disponible = Cantidades[i] * Minutos_Por_Semana
        prob += tiempo_requerido <= tiempo_disponible, f"Restriccion_maquina_{i}_{t}"

# Resolver el problema
prob.solve()


# Resultados

print("Status:", p.LpStatus[prob.status])
print("Costo Total: ", p.value(prob.objective))
resultados = {
    "Semana": list(range(1, tiempo + 1)),
    "Compra_Tela": [Tela[t].varValue for t in range(tiempo)],
    "Compra_Hilo": [Hilo[t].varValue for t in range(tiempo)],
    "Compra_Lamina": [Lamina[t].varValue for t in range(tiempo)],
    "Produccion_PT": [PT[t].varValue for t in range(tiempo)],
    "Inv_Tela": [Inventario[0][t].varValue for t in range(tiempo)],
    "Inv_Hilo": [Inventario[1][t].varValue for t in range(tiempo)],
    "Inv_Lamina": [Inventario[2][t].varValue for t in range(tiempo)],
    "Inv_PT": [Inventario[3][t].varValue for t in range(tiempo)],
}

df_resultados = pd.DataFrame(resultados)


df_resultados['Pronostico'] = Pronostico

# Grafico de lineas de producción y pronóstico

fig_line = px.line(
    df_resultados,
    x='Semana',
    y=['Produccion_PT', 'Pronostico'],
    title='Producción de Producto Terminado vs. Pronóstico de Demanda',
    labels={
        'value': 'Unidades',
        'variable': 'Leyenda'
    },
    markers=True
)
fig_line.data[0].name = 'Producción PT'
fig_line.data[1].name = 'Pronóstico'
fig_line.show()

# Tabla de resultados

fig_table = go.Figure(data=[go.Table(
    header=dict(values=list(df_resultados.columns),
                fill_color='paleturquoise',
                align='left'),
    cells=dict(values=[df_resultados[col] for col in df_resultados.columns],
               fill_color='lavender',
               align='left',
               # Formatear números en la tabla de plotly
               format=[None] + [',.0f'] * (len(df_resultados.columns)-1))
)])

fig_table.update_layout(
    title_text="Plan de abastecimiento y Producción",
    title_x=0.5
)
fig_table.show()

# Tablas de restricciones

## Restricción de Espacio de Almacenamiento
espacio_ocupado_vals = [
    (Inventario[0][t].varValue * Volumenes[0] +
     Inventario[1][t].varValue * Volumenes[1] +
     Inventario[2][t].varValue * Volumenes[2] +
     Inventario[3][t].varValue * Volumenes[3])
    for t in range(tiempo)
]
espacio_disponible_vals = [espacio_maximo[t] for t in range(tiempo)]

df_espacio = pd.DataFrame(
    [espacio_ocupado_vals, espacio_disponible_vals],
    columns=[f"Semana {t+1}" for t in range(tiempo)],
    index=['Espacio Ocupado [m3]', 'Espacio Disponible [m3]']
)

## Restricciones de Capacidad de Máquinas
maquina_nombres = ['Uso Corte CNC', 'Uso Corte Hilo', 'Uso Corte Plasma', 'Uso Ensamble']
dataframes_maquinas = []

for i in range(4): # Para cada máquina
    tiempo_requerido_vals = [PT[t].varValue * Tiempos[i] for t in range(tiempo)]
    tiempo_disponible_vals = [Cantidades[i] * Minutos_Por_Semana for t in range(tiempo)]
    
    df_maquina = pd.DataFrame(
        [tiempo_requerido_vals, tiempo_disponible_vals],
        columns=[f"Semana {t+1}" for t in range(tiempo)],
        index=['Tiempo Requerido [min]', 'Tiempo Disponible [min]']
    )
    dataframes_maquinas.append((maquina_nombres[i], df_maquina))

# Función para colorear
def highlight_usage(df):
    def color_row(row):
        # Tomamos la fila de 'requerido/ocupado' y 'disponible'
        usage = df.iloc[0]
        capacity = df.iloc[1]
        
        # Calculamos el ratio de uso
        ratio = usage / capacity
        
        # Aplicamos color basado en el ratio
        # 'background-color: ...' se aplica a la fila de uso
        # '' (vacío) se aplica a la fila de capacidad
        styles = pd.DataFrame('', index=df.index, columns=df.columns)
        styles.iloc[0] = ratio.apply(
            lambda x: 'background-color: #FF6961' if x >= 0.99 else ('background-color: #FDFD96' if x > 0.90 else 'background-color: #90EE90')
        )
        return styles
        
    return df.style.format("{:,.2f}").set_caption(df.name).apply(color_row, axis=None)


print("--- Tablas de Restricciones (Uso vs. Capacidad) ---\n")
print("Rojo: Uso al 99% o más de la capacidad (cuello de botella)")
print("Amarillo: Uso entre 90% y 99% de la capacidad (alto)")
print("Verde: Uso por debajo del 90% de la capacidad (holgura)\n")


df_espacio.name = "Restricción de Espacio"
display(highlight_usage(df_espacio)) # 'display()' es mejor para stylers en notebooks

for nombre, df in dataframes_maquinas:
    df.name = f"Restricción de Máquina: {nombre}"
    display(highlight_usage(df)) # Muestra cada tabla de máquina

# Exportar resultados a CSV y HTML
df_resultados.to_csv('resultados_plan_abastecimiento.csv', index=False)
fig_line.write_html('produccion_vs_pronostico.html')
fig_table.write_html('tabla_resultados.html')
# Guardar las tablas de restricciones
df_espacio.to_csv('restriccion_espacio.csv')
for nombre, df in dataframes_maquinas:
    df.to_csv(f'restriccion_maquina_{nombre.replace(" ", "_").lower()}.csv', index=False)




Status: Optimal
Costo Total:  377474.1800348135


--- Tablas de Restricciones (Uso vs. Capacidad) ---

Rojo: Uso al 99% o más de la capacidad (cuello de botella)
Amarillo: Uso entre 90% y 99% de la capacidad (alto)
Verde: Uso por debajo del 90% de la capacidad (holgura)



,Semana 1,Semana 2,Semana 3,Semana 4,Semana 5,Semana 6,Semana 7,Semana 8,Semana 9,Semana 10,Semana 11
Espacio Ocupado [m3],62.22,54.00,107.99,102.08,89.75,81.41,73.08,68.75,56.41,28.08,0.00
Espacio Disponible [m3],108.00,108.00,108.00,124.20,124.20,124.20,124.20,124.20,124.20,124.20,124.20


,Semana 1,Semana 2,Semana 3,Semana 4,Semana 5,Semana 6,Semana 7,Semana 8,Semana 9,Semana 10,Semana 11
Tiempo Requerido [min],"13,360.00","8,640.00","13,360.00","13,360.00","13,360.00","13,360.00","13,360.00","13,360.00","13,360.00","13,360.00","3,120.00"
Tiempo Disponible [min],"17,280.00","17,280.00","17,280.00","17,280.00","17,280.00","17,280.00","17,280.00","17,280.00","17,280.00","17,280.00","17,280.00"


,Semana 1,Semana 2,Semana 3,Semana 4,Semana 5,Semana 6,Semana 7,Semana 8,Semana 9,Semana 10,Semana 11
Tiempo Requerido [min],"16,700.00","10,800.00","16,700.00","16,700.00","16,700.00","16,700.00","16,700.00","16,700.00","16,700.00","16,700.00","3,900.00"
Tiempo Disponible [min],"17,280.00","17,280.00","17,280.00","17,280.00","17,280.00","17,280.00","17,280.00","17,280.00","17,280.00","17,280.00","17,280.00"


,Semana 1,Semana 2,Semana 3,Semana 4,Semana 5,Semana 6,Semana 7,Semana 8,Semana 9,Semana 10,Semana 11
Tiempo Requerido [min],"40,080.00","25,920.00","40,080.00","40,080.00","40,080.00","40,080.00","40,080.00","40,080.00","40,080.00","40,080.00","9,360.00"
Tiempo Disponible [min],"43,200.00","43,200.00","43,200.00","43,200.00","43,200.00","43,200.00","43,200.00","43,200.00","43,200.00","43,200.00","43,200.00"


,Semana 1,Semana 2,Semana 3,Semana 4,Semana 5,Semana 6,Semana 7,Semana 8,Semana 9,Semana 10,Semana 11
Tiempo Requerido [min],"133,600.00","86,400.00","133,600.00","133,600.00","133,600.00","133,600.00","133,600.00","133,600.00","133,600.00","133,600.00","31,200.00"
Tiempo Disponible [min],"133,920.00","133,920.00","133,920.00","133,920.00","133,920.00","133,920.00","133,920.00","133,920.00","133,920.00","133,920.00","133,920.00"
